In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16

# 1. Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 2. Normalize images
x_train = x_train / 255.0
x_test = x_test / 255.0

# 3. One-hot encode labels
y_train = tf.keras.utils.to_categorical(y_train,10)
y_test = tf.keras.utils.to_categorical(y_test,10)

# 4. Resize CIFAR images (32x32 → 224x224 for VGG)
# The previous approach caused an Out Of Memory (OOM) error because resizing all 50,000 images
# to 224x224 pixels at once consumes a large amount of GPU memory.
# To fix this, we will use `tf.data.Dataset` to resize images on-the-fly, in batches.

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 64 # Use the same batch size as intended for training

def preprocess_image(image, label):
    image = tf.image.resize(image, (IMG_HEIGHT, IMG_WIDTH))
    return image, label

# Create tf.data.Dataset objects for training and validation
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# Apply preprocessing (resize) and batch the datasets
train_dataset = train_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# 5. Load pretrained VGG16 (without classifier)
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3) # Use the defined IMG_HEIGHT and IMG_WIDTH
)

# 6. Freeze convolution layers
for layer in base_model.layers:
    layer.trainable = False

# 7. Add new classifier
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(10, activation='softmax')(x)

# 8. Create model
model = models.Model(inputs=base_model.input, outputs=output)

# 9. Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 10. Train model
history = model.fit(
    train_dataset, # Use the batched and preprocessed training dataset
    epochs=10,
    # batch_size=64, # Remove batch_size as it's handled by the dataset
    validation_data=test_dataset # Use the batched and preprocessed validation dataset
)

# 11. Evaluate
test_loss, test_acc = model.evaluate(test_dataset) # Use the batched and preprocessed test dataset
print("Test Accuracy:", test_acc)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 352s 419ms/step - accuracy: 0.3042 - loss: 1.9343 - val_accuracy: 0.4980 - val_loss: 1.4457
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 288s 369ms/step - accuracy: 0.4907 - loss: 1.4526 - val_accuracy: 0.5532 - val_loss: 1.2857
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 288s 368ms/step - accuracy: 0.5306 - loss: 1.3291 - val_accuracy: 0.5771 - val_loss: 1.2091
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 288s 368ms/step - accuracy: 0.5554 - loss: 1.2631 - val_accuracy: 0.5936 - val_loss: 1.1733
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 288s 368ms/step - accuracy: 0.5724 - loss: 1.2242 - val_accuracy: 0.6085 - val_loss: 1.1272
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 288s 369ms/step - accuracy: 0.5810 - loss: 1.1925 - val_accuracy: 0.6124 - val_loss: 1.1110
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 289s 370ms/step - accuracy: 0.5919 - loss: 1.1726 - val_accuracy: 0.6196 - val_loss: 1.0879
Epoch 8/10
782/782 ━━━━━━